# REWIND M2 Scaling100 재현용
실제 실행은 완료됐습니다. 기존 결과 회수 목적이면 이 notebook으로 학습을 다시 실행하지 마세요. 새 재현 실행에만 사용합니다. 무료 T4를 선택하고 제공된 M2_scaling100_gpu_input.zip을 업로드하세요. 소스6ccf0ab 고정, 두 군400 updates, 기존 결과 덮어쓰기 금지.


In [ ]:
import torch, platform
print(platform.python_version(), torch.__version__)
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name()
print('GPU:', torch.cuda.get_device_name())

In [ ]:
from pathlib import Path
import subprocess, os, base64, hashlib, zipfile, io
REPO=Path('/content/rewind_scaling100')
assert not REPO.exists(), 'Use a new isolated directory; do not overwrite a run.'
subprocess.run(['git','clone','--quiet','https://github.com/darkkyoung/REWIND-like-prototype.git',str(REPO)],check=True)
subprocess.run(['git','checkout','6ccf0ab650a225ec0c823b9ff030ce19c5a7dc84'],cwd=REPO,check=True)
os.chdir(REPO)
from google.colab import files
uploaded=files.upload()
assert set(uploaded)=={'M2_scaling100_gpu_input.zip'}
payload=uploaded['M2_scaling100_gpu_input.zip']
assert hashlib.sha256(payload).hexdigest()=='9145430464cbd44b1bb220ea93545b0015a7f8a4d02e63a940b489618bd623f3'
input_dir=Path('/content/m2_scaling100_input');input_dir.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    assert all(Path(n).name==n for n in z.namelist())
    z.extractall(input_dir)
print('INPUT_READY',len(payload),'bytes',hashlib.sha256(payload).hexdigest())


In [ ]:
import subprocess, sys, time
from pathlib import Path
prefix=Path('experiments/final_m2/dataset_scaling_100')
checks=[]
for pattern in ('test_m2.py','test_final_metrics.py'):
    check=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p',pattern,'-v'],cwd=REPO,capture_output=True,text=True)
    print(check.stdout,check.stderr,flush=True)
    checks.append(check.stdout+check.stderr)
    assert check.returncode==0
out=Path('/content/m2_scaling100_t4_results')
assert not out.exists(), 'This experiment has already started; recover it, do not rerun.'
started=time.monotonic()
with Path('/content/m2_scaling100_console.log').open('x') as logfile:
    proc=subprocess.Popen([sys.executable,'-u',str(prefix/'run_experiment.py'),'--input-dir',str(input_dir),'--output',str(out)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,cwd=REPO)
    for line in proc.stdout:
        print(line,end='',flush=True);logfile.write(line);logfile.flush()
    code=proc.wait()
(out/'colab_regression_tests.log').write_text('\n'.join(checks))
print('RETURN_CODE',code,'ELAPSED_SECONDS',time.monotonic()-started)
assert code==0, 'Preserve failure and checkpoints; no automatic retraining.'


In [ ]:
# Recovery/export only: no training or inference is executed here.
import json,zipfile,hashlib,base64
from pathlib import Path
from IPython.display import display,HTML
out=Path('/content/m2_scaling100_t4_results')
report=json.loads((out/'evaluation.json').read_text())
assert report['execution_status']=='PASS'
print('EXECUTION',report['execution_status'],'QUALITY',report['quality']['status'])
for r in report['results']:
    if r['split']=='opening130':
        print({k:r[k] for k in ('model','global_joint_error_raw','root_relative_error_22_raw','frame_displacement_error_raw','motion_magnitude_ratio_pct','root_relative_motion_ratio_pct','bone_length_ratio_pct','joint_spread_ratio_pct','pelvis_position_error_raw')})
archive=Path('/content/REWIND_M2_Scaling100_T4_raw.zip')
assert not archive.exists()
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()):
        if p.is_file():z.write(p,p.name)
    z.write('/content/m2_scaling100_console.log','console.log')
raw=archive.read_bytes()
print('RAW_ZIP_BYTES',len(raw),'SHA256',hashlib.sha256(raw).hexdigest())
display(HTML('<a download="REWIND_M2_Scaling100_T4_raw.zip" href="data:application/zip;base64,'+base64.b64encode(raw).decode()+'">Download verified scaling100 raw T4 results</a>'))
